In [ ]:

"""
即時麥克風錄音 → 提頻（+4 semitones）→ 播放時同步顯示波動頻譜圖

使用方式：
    pip install sounddevice librosa soundfile numpy scipy matplotlib
    python voice_pitch_spectrum_animated.py

操作流程：
    1. 程式啟動後先錄音（預設 5 秒）
    2. 自動提頻
    3. 播放原始 & 提頻後音檔，頻譜圖即時波動
"""

import sounddevice as sd
import soundfile as sf
import librosa
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from scipy.signal import butter, sosfilt, sosfilt_zi
import threading
import queue
import sys

# ─── 參數設定 ────────────────────────────────────────────────
RECORD_SECONDS  = 5
RATE            = 44100
CHANNELS        = 1
CHUNK           = 2048          # 每次分析的 sample 數（約 46ms）
HOP             = 1024          # 每次前進的 sample 數（控制更新頻率）

PITCH_SEMITONES = 4
CUTOFF_FREQ     = 80.0
NUM_BINS        = 24
FREQ_MIN        = 20.0
FREQ_MAX        = 20000.0

RAW_FILE        = "recording_raw.wav"
SHIFTED_FILE    = "recording_shifted.wav"

# ─── 建立高通濾波器 ──────────────────────────────────────────
sos = butter(N=6, Wn=CUTOFF_FREQ, btype="highpass", fs=RATE, output="sos")

# ─── 建立對數頻段索引 ────────────────────────────────────────
x_freq      = np.fft.rfftfreq(CHUNK, 1.0 / RATE)
log_edges   = np.logspace(np.log10(FREQ_MIN), np.log10(FREQ_MAX), NUM_BINS + 1)
bin_centers = np.sqrt(log_edges[:-1] * log_edges[1:])

band_indices = []
for i in range(NUM_BINS):
    idx = np.where((x_freq >= log_edges[i]) & (x_freq < log_edges[i + 1]))[0]
    band_indices.append(idx if len(idx) > 0 else np.array([0]))

xtick_labels = []
for f in bin_centers:
    xtick_labels.append(f"{int(f)}" if f < 1000 else f"{f/1000:.1f}k")
display_labels = [l if i % 2 == 0 else "" for i, l in enumerate(xtick_labels)]

window_fn = np.hanning(CHUNK)

# ─── 決定箱子顏色 ────────────────────────────────────────────
def bin_color(freq):
    if freq < CUTOFF_FREQ:      return "#444444"
    elif freq <= 255:           return "#ffaa00"
    elif freq <= 8000:          return "#00e5ff"
    else:                       return "#2a5a5a"

BIN_COLORS = [bin_color(f) for f in bin_centers]

# ─── 計算單一 chunk 的每頻段中位數（用於即時動畫）───────────
def chunk_to_medians(seg: np.ndarray, zi):
    filtered, zi_new = sosfilt(sos, seg, zi=zi)
    windowed = filtered * window_fn
    fft_result = np.fft.rfft(windowed)
    magnitude  = np.abs(fft_result) / (CHUNK / 2)
    db_data    = 20 * np.log10(magnitude + 1e-10)
    medians = np.array([np.median(db_data[idx]) for idx in band_indices])
    return medians, zi_new

# ─── 錄音 ────────────────────────────────────────────────────
def record_audio():
    print(f"🎙  開始錄音（{RECORD_SECONDS} 秒），請說話...")
    audio = sd.rec(int(RECORD_SECONDS * RATE), samplerate=RATE,
                   channels=CHANNELS, dtype="float32")
    sd.wait()
    audio = audio.squeeze()
    sf.write(RAW_FILE, audio, RATE)
    print(f"✅ 錄音完成 → {RAW_FILE}")
    return audio

# ─── 提頻 ────────────────────────────────────────────────────
def pitch_shift(audio: np.ndarray):
    print(f"🎵 提頻中（+{PITCH_SEMITONES} 半音）...")
    shifted = librosa.effects.pitch_shift(
        audio.astype(np.float32), sr=RATE, n_steps=PITCH_SEMITONES
    )
    sf.write(SHIFTED_FILE, shifted, RATE)
    print(f"✅ 提頻完成 → {SHIFTED_FILE}")
    return shifted

# ─── 播放執行緒：邊播放邊把 chunk 推入 queue ─────────────────
def playback_thread(audio: np.ndarray, q: queue.Queue, label: str):
    """
    用 callback 播放音訊。每播放一個 chunk 就把它放進 queue，
    讓主執行緒的繪圖迴圈取用。
    """
    pos = [0]   # 用 list 讓 closure 可以修改

    def callback(outdata, frames, time, status):
        start = pos[0]
        end   = start + frames
        chunk = audio[start:end]
        if len(chunk) < frames:
            chunk = np.pad(chunk, (0, frames - len(chunk)))
        outdata[:, 0] = chunk
        pos[0] = end

        # 推入繪圖用 queue（不阻塞，滿了就捨棄舊的）
        if not q.full():
            q.put_nowait(chunk.copy())

    total_frames = len(audio)
    with sd.OutputStream(
        samplerate=RATE, channels=1, blocksize=CHUNK,
        dtype="float32", callback=callback
    ):
        while pos[0] < total_frames:
            sd.sleep(10)

    q.put(None)   # 結束信號
    print(f"  [{label}] 播放完畢")

# ─── 主繪圖迴圈 ──────────────────────────────────────────────
def animate_spectrum(audio: np.ndarray, title: str, color_accent: str):
    """播放音訊並即時更新頻譜。"""
    q = queue.Queue(maxsize=4)

    # 啟動播放執行緒
    t = threading.Thread(target=playback_thread, args=(audio, q, title), daemon=True)
    t.start()

    # ── 建立圖表 ──
    plt.ion()
    fig = plt.figure(figsize=(13, 5.5), facecolor="#1a1a1a")
    fig.suptitle(title, color="white", fontsize=13, y=0.97)
    ax = fig.add_subplot(111)
    ax.set_facecolor("#1a1a1a")
    ax.set_ylim(-100, 0)
    ax.set_xlim(0, NUM_BINS + 1)
    ax.set_ylabel("Relative Intensity (dBFS)", color="#aaaaaa", fontsize=10)
    ax.set_xlabel("Frequency (Hz)", color="#aaaaaa", fontsize=10)
    ax.tick_params(colors="#aaaaaa")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
    ax.set_xticks(range(1, NUM_BINS + 1))
    ax.set_xticklabels(display_labels, color="#aaaaaa", rotation=45, fontsize=8)

    # 初始化長條（用 bar chart 模擬即時波動，比 boxplot 更流暢）
    x_pos = np.arange(1, NUM_BINS + 1)
    bars  = ax.bar(x_pos, [-60] * NUM_BINS, width=0.7, color=BIN_COLORS,
                   alpha=0.85, zorder=3)

    # 格線
    ax.yaxis.grid(True, color="#2a2a2a", linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)

    # 進度文字
    progress_text = ax.text(
        0.99, 0.96, "▶ 0.0s", transform=ax.transAxes,
        color=color_accent, fontsize=9, ha="right", va="top",
        fontfamily="monospace"
    )

    # 平滑用：儲存前幾幀取平均（減少閃爍）
    smoothed = np.full(NUM_BINS, -80.0)
    SMOOTH   = 0.55   # 越大越平滑（0~1）

    zi       = sosfilt_zi(sos) * 0.0
    elapsed  = 0.0
    frame_dt = CHUNK / RATE   # 每 chunk 的秒數

    while True:
        try:
            seg = q.get(timeout=0.5)
        except queue.Empty:
            continue

        if seg is None:
            break

        # 補齊不足一個 chunk 的尾段
        if len(seg) < CHUNK:
            seg = np.pad(seg, (0, CHUNK - len(seg)))

        medians, zi = chunk_to_medians(seg, zi)

        # 指數移動平均平滑
        smoothed = SMOOTH * smoothed + (1 - SMOOTH) * medians

        # 更新長條高度（從 -100 往上）
        for bar, val in zip(bars, smoothed):
            bar.set_height(val + 100)   # ax ylim 從 -100 開始
            bar.set_y(-100)

        elapsed += frame_dt
        progress_text.set_text(f"▶ {elapsed:.1f}s")

        fig.canvas.draw()
        fig.canvas.flush_events()

    plt.ioff()
    plt.show(block=False)
    plt.pause(0.5)
    t.join()

# ─── 主程式 ──────────────────────────────────────────────────
def main():
    # 步驟 1：錄音
    raw_audio = record_audio()

    # 步驟 2：提頻
    shifted_audio = pitch_shift(raw_audio)

    print("\n▶  即將播放「原始錄音」並顯示頻譜...")
    input("  按 Enter 開始 → ")
    animate_spectrum(raw_audio, "頻譜分析 — 原始錄音 (Raw)", "#ffaa00")

    print("\n▶  即將播放「提頻後」錄音並顯示頻譜...")
    input("  按 Enter 開始 → ")
    animate_spectrum(shifted_audio,
                     f"頻譜分析 — 提頻後 (+{PITCH_SEMITONES} 半音)", "#00e5ff")

    print("\n✅ 完成！")

if __name__ == "__main__":
    main()